In [26]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [27]:
master = pd.read_csv(
    'cleaned_master_dataset.csv'
)

print(
    "Dataset Loaded Successfully"
)

Dataset Loaded Successfully


In [28]:
master.shape

(82365, 39)

# Convert Purchase Date

In [29]:
master[
    'order_purchase_timestamp'
] = pd.to_datetime(
    master[
        'order_purchase_timestamp'
    ],
    format='mixed',
    dayfirst=True
)

# Create Monthly Revenue Table

In [30]:
master['Year_Month'] = (
    master[
        'order_purchase_timestamp'
    ]
    .dt.to_period('M')
)

monthly_revenue = (
    master
    .groupby(
        'Year_Month'
    )['revenue']
    .sum()
    .reset_index()
)

monthly_revenue.head()

,Year_Month,revenue
0,2016-09,207.86
1,2016-10,28266.76
2,2016-12,10.90
3,2017-01,75760.07
4,2017-02,140525.16


# Prepare Time Index

In [31]:
monthly_revenue['Month_Index'] = range(
    len(monthly_revenue)
)

monthly_revenue.head()

,Year_Month,revenue,Month_Index
0,2016-09,207.86,0
1,2016-10,28266.76,1
2,2016-12,10.90,2
3,2017-01,75760.07,3
4,2017-02,140525.16,4


# Train Revenue Forecast Model

In [32]:
X = monthly_revenue[
    ['Month_Index']
]

y = monthly_revenue[
    'revenue'
]

model = LinearRegression()

model.fit(
    X,
    y
)

print(
    "Forecast Model Trained"
)

Forecast Model Trained


# Forecast Next 6 Months

In [33]:
future_months = pd.DataFrame({
    'Month_Index': range(
        len(monthly_revenue),
        len(monthly_revenue) + 6
    )
})

future_months['Forecast_Revenue'] = (
    model.predict(
        future_months
    )
)

future_months

,Month_Index,Forecast_Revenue
0,25,533530.221500
1,26,549279.486631
2,27,565028.751762
3,28,580778.016892
4,29,596527.282023
5,30,612276.547154


# Revenue Growth Rate Analysis

In [34]:
monthly_revenue['Growth_Rate_%'] = (
    monthly_revenue['revenue']
    .pct_change() * 100
)

monthly_revenue.head()

,Year_Month,revenue,Month_Index,Growth_Rate_%
0,2016-09,207.86,0,NaN
1,2016-10,28266.76,1,13498.941595
2,2016-12,10.90,2,-99.961439
3,2017-01,75760.07,3,694946.513761
4,2017-02,140525.16,4,85.487104


# Revenue Trend Classification

In [35]:
avg_growth = monthly_revenue[
    'Growth_Rate_%'
].mean()

if avg_growth > 0:
    print("Positive Revenue Trend")
else:
    print("Negative Revenue Trend")

Positive Revenue Trend


# Forecast Growth Percentage

In [36]:
if current_revenue != 0:

    growth_forecast = (
        (forecast_revenue - current_revenue)
        / current_revenue
    ) * 100

    print(
        "Forecast Growth:",
        round(growth_forecast,2),
        "%"
    )

else:

    print(
        "Current revenue is zero. Growth percentage cannot be calculated."
    )

Current revenue is zero. Growth percentage cannot be calculated.


In [37]:
average_revenue = monthly_revenue[
    'revenue'
].mean()

forecast_revenue = future_months[
    'Forecast_Revenue'
].mean()

growth_forecast = (
    (
        forecast_revenue
        -
        average_revenue
    )
    /
    average_revenue
) * 100

print(
    "Forecast Revenue Growth:",
    round(growth_forecast,2),
    "%"
)

Forecast Revenue Growth: 74.25 %


# Revenue Risk Alert System

In [38]:
def risk_alert(growth):

    if growth < -10:
        return "High Risk"

    elif growth < 0:
        return "Medium Risk"

    else:
        return "Healthy"

future_months[
    'Risk_Level'
] = future_months[
    'Forecast_Revenue'
].pct_change().fillna(0).apply(
    risk_alert
)

future_months

,Month_Index,Forecast_Revenue,Risk_Level
0,25,533530.221500,Healthy
1,26,549279.486631,Healthy
2,27,565028.751762,Healthy
3,28,580778.016892,Healthy
4,29,596527.282023,Healthy
5,30,612276.547154,Healthy


# Revenue Opportunity Score

In [39]:
future_months[
    'Opportunity_Score'
] = (
    future_months[
        'Forecast_Revenue'
    ]
    /
    future_months[
        'Forecast_Revenue'
    ].max()
    *100
).round(2)

# Best Forecast Month

In [40]:
future_months.loc[
    future_months[
        'Forecast_Revenue'
    ].idxmax()
]

,5
Month_Index,30
Forecast_Revenue,612276.547154
Risk_Level,Healthy
Opportunity_Score,100.0


# Dashboard Forecast Dataset

In [41]:
forecast_dashboard = future_months.copy()

forecast_dashboard.head()

,Month_Index,Forecast_Revenue,Risk_Level,Opportunity_Score
0,25,533530.221500,Healthy,87.14
1,26,549279.486631,Healthy,89.71
2,27,565028.751762,Healthy,92.28
3,28,580778.016892,Healthy,94.86
4,29,596527.282023,Healthy,97.43


In [42]:
forecast_dashboard.to_csv(
    'forecast_dashboard.csv',
    index=False
)

# Revenue Target Achievement Simulator

In [43]:
target_revenue = 1000000

future_months[
    'Target_Status'
] = np.where(
    future_months[
        'Forecast_Revenue'
    ] >= target_revenue,
    'Target Achieved',
    'Below Target'
)

future_months

,Month_Index,Forecast_Revenue,Risk_Level,Opportunity_Score,Target_Status
0,25,533530.221500,Healthy,87.14,Below Target
1,26,549279.486631,Healthy,89.71,Below Target
2,27,565028.751762,Healthy,92.28,Below Target
3,28,580778.016892,Healthy,94.86,Below Target
4,29,596527.282023,Healthy,97.43,Below Target
5,30,612276.547154,Healthy,100.00,Below Target


# Forecast Summary

In [44]:
forecast_summary = pd.DataFrame({
    'Metric': [
        'Average Historical Revenue',
        'Average Forecast Revenue',
        'Forecast Growth %'
    ],
    'Value': [
        average_revenue,
        future_months[
            'Forecast_Revenue'
        ].mean(),
        growth_forecast
    ]
})

forecast_summary

,Metric,Value
0,Average Historical Revenue,328789.774800
1,Average Forecast Revenue,572903.384327
2,Forecast Growth %,74.246107


# Revenue Forecast

In [45]:
future_months.to_csv(
    'revenue_forecast.csv',
    index=False
)

# Monthly Revenue Trend

In [46]:
monthly_revenue.to_csv(
    'monthly_revenue_trend.csv',
    index=False
)

# Forecast Dashboard

In [47]:
forecast_dashboard.to_csv(
    'forecast_dashboard.csv',
    index=False
)

# Forecast Summary

In [48]:
forecast_summary.to_csv(
    'forecast_summary.csv',
    index=False
)

In [49]:
import os

os.listdir()

['.config',
 'forecast_dashboard.csv',
 'cleaned_master_dataset.csv',
 'revenue_forecast.csv',
 'forecast_summary.csv',
 'monthly_revenue_trend.csv',
 'sample_data']